In [2]:
import numpy as np
import pandas as pd

### Load Data

In [214]:
data = pd.read_csv('bps-elem-data.csv')
data.head()

,School,ID,Address,City_State_Zip,Grades,Level,Neighborhood,Capacity,Utilization (5 year avg),Building_Exp,...,Pct_Neighborhood,Pct_Walkzone,K1_First_Choice,K1_Top_3,K2_First_Choice,K2_Top_3,Private_Dollar_per_Student,Pct_Econ_Disadv,Opp_Index,MCAS_Tier
0,Adams Elementary School,4361,165 Webster St.,"East Boston, MA 02128",K0-6,Elementary,East Boston,236,102%,2,...,96.68%,59.72%,46.36%,69.35%,36.81%,63.19%,$89,50.0,0.429,3.000
1,Alighieri Dante Montessori School,4321,37 Gove St.,"East Boston, MA 02128",K0-6,Elementary,East Boston,80,134%,3,...,100.00%,75.76%,80.95%,90.48%,51.85%,66.67%,$60,25.0,0.428,2.125
2,Baldwin Early Learning Pilot Academy,4621,121 Corey Rd.,"Brighton, MA 02135",K0-1,Elementary,Allston/Brighton,246,67%,2,...,90.28%,56.94%,61.03%,70.94%,53.04%,77.14%,$822,33.0,0.385,2.250
3,Bates Elementary School,4081,426 Beech St.,"Roslindale, MA 02131",K0-6,Elementary,Roslindale,164,165%,0,...,51.74%,53.82%,90.20%,98.04%,78.07%,93.57%,$191,46.0,0.316,2.000
4,Beethoven Elementary School,4030,5125 Washington St.,"West Roxbury, MA 02132",K1-2,Elementary,West Roxbury,237,115%,4,...,42.44%,32.84%,89.94%,100.00%,67.41%,89.55%,$66,38.0,0.304,2.000


### Cleaning Formatting

In [215]:
cols = ['Asian', 'Black', 'Hispanic', 'MultiRacial', 'Native_American', 'Pacific_Islander', 'White',
        'AA_K1', 'AA_K2', 'Pct_Neighborhood', 'Pct_Walkzone', 'K1_First_Choice', 'K1_Top_3', 'K2_First_Choice', 'K2_Top_3']

for col in cols:
    data[col] = data[col].str.rstrip('%').astype('float') / 100

### Synthetic Data

#### Use GIS data to get Neighborhood Centers (Lat, Lon)

In [182]:
import geopandas as gpd

gdf = gpd.read_file("BPDA_Neighborhoods.shp")

gdf = gdf.set_crs(epsg=2249)
gdf = gdf.to_crs(epsg=4326)

gdf["centroid"] = gdf.geometry.centroid
gdf["lat"] = gdf.centroid.y
gdf["lon"] = gdf.centroid.x

C:\Users\Julie\AppData\Local\Temp\ipykernel_220\3711789459.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid"] = gdf.geometry.centroid
C:\Users\Julie\AppData\Local\Temp\ipykernel_220\3711789459.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.centroid.y
C:\Users\Julie\AppData\Local\Temp\ipykernel_220\3711789459.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.centroid.x


In [183]:
gdf = gdf[['lat', 'lon']]

neighborhoods = pd.read_csv('BPDA_Neighborhoods.csv')
neighborhoods = neighborhoods[['name']]

# Make DF with name, longitude, latitude
neighborhoods = neighborhoods.join(gdf)

# Remove neighborhoods not specifed in dataset
drop_idx = [10, 4, 14, 17, 6, 16, 25, 1, 5, 3, 7, 22, 13]
neighborhoods = neighborhoods.drop(drop_idx)

# Rename neighborhoods to match dataset
neighborhoods.at[24, 'name'] = 'Allston/Brighton'
neighborhoods.at[2, 'name'] = 'Mission Hill/Jamaica Plain'

# Reset indexing
neighborhoods = neighborhoods.reset_index(drop=True)


In [184]:
def haversine(coords):
    R = 3959  # Earth radius in mi
    
    lat = np.radians(coords[:, 0])
    lon = np.radians(coords[:, 1])
    
    lat1 = lat[:, np.newaxis]
    lat2 = lat[np.newaxis, :]
    
    lon1 = lon[:, np.newaxis]
    lon2 = lon[np.newaxis, :]
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [185]:
coords = neighborhoods[['lat', 'lon']].to_numpy()
matrix = haversine(coords)

dist_df = pd.DataFrame(
    matrix,
    index=neighborhoods['name'],
    columns=neighborhoods['name']
)

#### Use Softmax to Create Exponentially Weighted Probabilities (if a student is from a different neighborhood)

In [186]:
def distance_to_probs(dist_row, lam=1.0, exclude_self=True):
    d = dist_row.copy()
    
    if exclude_self:
        d = d.drop(d.index[d == 0])  # remove self-distance
    
    weights = np.exp(-lam * d)
    probs = weights / weights.sum()
    
    return probs

#probs = distance_to_probs(dist_df.loc['Dorchester'], lam=2)  # choosing lambda (arbitrarily?) - higher lambda = more emphasis on closeness

#### Cleaning function for formatting issue

In [217]:
def clean_name(s):
    return (
        s.strip()
         .replace(" / ", "/")
         .replace(" /", "/")
         .replace("/ ", "/")
    )

data['Neighborhood'] = data['Neighborhood'].apply(clean_name)
dist_df.index = dist_df.index.map(clean_name)
dist_df.columns = dist_df.columns.map(clean_name)

#### Function to generate data

Note: currently outputs the number of students with a randomly generated neighborhood per school

In [ ]:
def generate_students(df):
    students = []

    replacement_count = []
    for _, row in df.iterrows():    # iterate over the schools
        n = int(row['Enrollment'])  # number of students at the school

        # draw race distribution for the school from a multinomial distribution
        race_counts = np.random.multinomial(n,
            [
                row['Asian'],
                row['Black'],
                row['Hispanic'],
                row['MultiRacial'],
                row['Native_American'],
                row['Pacific_Islander'],
                row['White']
            ]
        )

        races = (
            ['Asian'] * race_counts[0] +
            ['Black'] * race_counts[1] +
            ['Hispanic'] * race_counts[2] +
            ['MultiRacial'] * race_counts[3] +
            ['Native_American'] * race_counts[4] +
            ['Pacific_Islander'] * race_counts[5] +
            ['White'] * race_counts[6]
        )

        np.random.shuffle(races) 

        count = 0
        for i in range(n):
            in_neighborhood = np.random.rand() < row['Pct_Neighborhood']  # assign if student lives in the neighborhood of the school
            econ = np.random.rand() < row['Pct_Econ_Disadv']              # assign if the stucent is economically disadvantaged
            

            if in_neighborhood:
                neighborhood = row['Neighborhood']
            else:
                probs = distance_to_probs(dist_df.loc[row['Neighborhood']], lam=2)
                neighborhood = np.random.choice(probs.index, p=probs.values)
                count += 1

            students.append({
                'school': row['School'],
                'race': races[i],
                'neighborhood': neighborhood,
                'econ_disadvantaged': econ
            })

        replacement_count.append({
            'School': row['School'],
            'Empirical Replacements': count
        })

    return students, replacement_count

In [226]:
students, replacement = generate_students(data)
replacements = pd.DataFrame(replacement)

In [228]:
true_oon = data[['School', 'Enrollment', 'Pct_Neighborhood']]
true_oon['True Replacements'] = true_oon['Enrollment'] - round(true_oon['Enrollment'] * true_oon['Pct_Neighborhood'])
true_oon = true_oon[['True Replacements']]

replacements = replacements.join(true_oon)
replacements

C:\Users\Julie\AppData\Local\Temp\ipykernel_220\2640449398.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  true_oon['True Replacements'] = true_oon['Enrollment'] - round(true_oon['Enrollment'] * true_oon['Pct_Neighborhood'])


,School,Empirical Replacements,True Replacements
0,Adams Elementary School,2,7.0
1,Alighieri Dante Montessori School,0,0.0
2,Baldwin Early Learning Pilot Academy,8,15.0
3,Bates Elementary School,134,141.0
4,Beethoven Elementary School,151,159.0
...,...,...,...
68,UP Academy Holland,320,332.0
69,Warren-Prescott K-8 School,107,96.0
70,Winship Elementary School,12,10.0
71,Winthrop Elementary School,91,91.0
